# Reproducibility package paths and historical outputs

Original source: `/data/hulei/HongtaoYu_Lab/Fetal_brain/Revision/GRN/bin100/robustness/01_counts50.ipynb`.
Original file SHA256: `afee116a7561d2f350c5c2ef3534f8d2c1d775207c753e8f3b33a2b5e0ba003a`.

**This notebook was not re-executed or retrained for this update. All execution counts and cell outputs below are preserved from the original notebook as historical records. The updated paths have not been runtime-validated.**

Run from the `reproducibility` package root, where `base = Path.cwd()` and `config.json` provides the runtime inputs. By default, the notebook reads the original external input matrix. New training results, temporary files, downsampled matrices, and QC files are written outside the package under `../reproducibility_work`. Original inputs, prior results, and package data are not written back.

Inference parameters, random seeds, constant-gene handling, and AUCell settings are unchanged. The two ranking databases use their exact configured file paths.


# T577 bin100: Five independent counts50 downsampling replicates
For each bin, sample `floor(total_counts / 2)` from all integer counts without replacement; detected-gene counts are unconstrained.
Seeds are 20261016–20261020. Retain all bins, gene columns, and spatial coordinates.
For rep01, recompute the sampled matrix and compare it entry by entry with the existing matrix, then reuse its existing GRN. Save new inputs for the other four replicates.
Source counts were reconstructed using `round(expm1(expr))`; this does not establish that they are raw molecule or UMI counts.

For this copy, rep01–rep05 all write to the external `generated_input` directory. Rep01 is still checked entry by entry against the historical `source_input` first. The preceding description and original cell outputs document historical execution. The sampling function, seeds, and count targets are unchanged.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

base = Path.cwd()
config = json.loads((base / "config.json").read_text())["training"]
source = sc.read_h5ad(base / config["full"]["input"])
source_sums = np.asarray(source.X.sum(axis=1, dtype=np.int64)).ravel()
(base / config["downsampling"]["output_dir"]).mkdir(parents=True, exist_ok=True)
(base / config["downsampling"]["qc"]).parent.mkdir(parents=True, exist_ok=True)


In [2]:
def thin_counts(matrix, seed):
    rng = np.random.default_rng(seed)
    values, columns = [], []
    pointers = np.zeros(matrix.shape[0] + 1, dtype=np.int64)
    for row, (left, right) in enumerate(zip(matrix.indptr[:-1], matrix.indptr[1:])):
        counts = matrix.data[left:right].astype(np.int64)
        draws = rng.multivariate_hypergeometric(counts, int(counts.sum()) // 2)
        keep = draws > 0
        values.append(draws[keep].astype(np.int32))
        columns.append(matrix.indices[left:right][keep])
        pointers[row + 1] = pointers[row] + int(keep.sum())
    return sparse.csr_matrix((np.concatenate(values), np.concatenate(columns), pointers),
                             shape=matrix.shape, dtype=np.int32)

In [3]:
rows = []
for run in config["runs"]:
    sampled = thin_counts(source.X, run["seed"])
    sums = np.asarray(sampled.sum(axis=1, dtype=np.int64)).ravel()
    if run["run"] == "rep01":
        existing = sc.read_h5ad(base / run["source_input"])
        differences = int((sampled != existing.X).nnz)
        del existing
        if differences:
            raise ValueError(f"rep01 historical matrix differs at {differences} entries")
    else:
        differences = np.nan
    data = source.copy()
    data.X = sampled
    data.obs["total_counts"] = sums
    data.obs["n_genes_by_counts"] = np.diff(sampled.indptr)
    data.var["total_counts"] = np.asarray(sampled.sum(axis=0, dtype=np.int64)).ravel()
    data.var["n_cells_by_counts"] = np.asarray((sampled > 0).sum(axis=0)).ravel()
    data.uns["source_counts_provenance"] = data.uns.pop("counts_provenance")
    data.uns["counts_provenance"] = dict(seed=run["seed"], source_file=config["full"]["input"],
        operation="per-bin multivariate hypergeometric sampling without replacement",
        target="floor(source_total_counts/2)", detected_gene_count_constraint="none",
        raw_umi_provenance_confirmed=False)
    data.write_h5ad(base / run["generated_input"], compression="gzip")
    del data
    rows.append(dict(run=run["run"], seed=run["seed"], n_bins=sampled.shape[0],
        n_genes=sampled.shape[1], total_counts=int(sums.sum()), nnz=sampled.nnz,
        count_target_mismatches=int(np.count_nonzero(sums != source_sums // 2)),
        source_bound_violations=int(np.count_nonzero((sampled - source.X).data > 0)),
        rep01_matrix_differences=differences, mean_detected_genes=float(np.diff(sampled.indptr).mean()),
        input=run["generated_input"]))
    del sampled
qc = pd.DataFrame(rows)
qc.to_csv(base / config["downsampling"]["qc"], sep="\t", index=False)
qc


,run,seed,n_bins,n_genes,total_counts,nnz,count_target_mismatches,source_bound_violations,rep01_matrix_differences,mean_detected_genes,input
0,rep01,20261016,117867,15541,135429514,82157995,0,0,0.0,697.039842,/data/hulei/HongtaoYu_Lab/Fetal_brain/Revision...
1,rep02,20261017,117867,15541,135429514,82156433,0,0,NaN,697.026589,/data/hulei/HongtaoYu_Lab/Fetal_brain/Revision...
2,rep03,20261018,117867,15541,135429514,82150103,0,0,NaN,696.972885,/data/hulei/HongtaoYu_Lab/Fetal_brain/Revision...
3,rep04,20261019,117867,15541,135429514,82157580,0,0,NaN,697.036321,/data/hulei/HongtaoYu_Lab/Fetal_brain/Revision...
4,rep05,20261020,117867,15541,135429514,82152858,0,0,NaN,696.996258,/data/hulei/HongtaoYu_Lab/Fetal_brain/Revision...
